<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/dinov2-embeddings/dinov2_retrieval_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DINOv2 Minifigure Retrieval Prototype

Evaluate DINOv2 as a drop-in replacement for the shipped SimCLR torso encoder.

**Runtime:** Select **Runtime → Change runtime type → GPU** (free T4 is enough; A100 finishes the full catalog in ~3 min).

## 1. Clone repo & install dependencies

In [ ]:
MODEL = "dinov2_vitl14"  # Options: dinov2_vits14, dinov2_vitb14, dinov2_vitl14

import os

if not os.path.exists('Bricky'):
    !git clone --depth 1 https://github.com/shribr/Bricky.git
else:
    print('Bricky/ already exists — skipping clone.')

os.chdir('Bricky')
print('CWD:', os.getcwd())

In [ ]:
!pip install -q -r Tools/dinov2-embeddings/requirements.txt

# Create output directories that the pipeline writes to
import os
for d in [
    'Tools/dinov2-embeddings/index',
    'Tools/dinov2-embeddings/reports',
    'Tools/dinov2-embeddings/eval_set',
]:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓ {d}')
print('Output directories ready.')

## 2. Build the evaluation set

Deterministic given the seed. Creates 200 figures × 8 variants = 1 600 query images.

In [ ]:
!python Tools/dinov2-embeddings/build_eval_set.py \
    --count 200 --variants-per-figure 8 --seed 42

## 3. Embed the catalog with DINOv2

Start with **ViT-S** (fastest, smallest bundle). If recall is too low, re-run with `dinov2_vitb14` or `dinov2_vitl14`.

In [ ]:
# MODEL is set in the setup cell above. Override here if needed:
# MODEL = "dinov2_vits14"
# MODEL = "dinov2_vitb14"
# MODEL = "dinov2_vitl14"
print(f"Using model: {MODEL}")

In [ ]:
!python Tools/dinov2-embeddings/embed_catalog.py \
    --model ${MODEL} \
    --out Tools/dinov2-embeddings/index/${MODEL}

## 4. Evaluate DINOv2 retrieval

In [ ]:
!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/${MODEL} \
    --report Tools/dinov2-embeddings/reports/${MODEL}.json

## 5. Score the shipped SimCLR index (baseline)

Requires `Tools/torso-embeddings/out/torso_encoder.pt` to be present.

In [ ]:
!python Tools/dinov2-embeddings/compare_existing.py \
    --report Tools/dinov2-embeddings/reports/simclr_shipped.json

## 6. Compare reports

Load both JSON reports and compare `recall@1`, `recall@5`, `recall@10`.

| recall@5 | Interpretation |
|----------|----------------|
| < 0.20   | Encoder is effectively blind — likely a crop/preprocessing bug |
| 0.20–0.50 | Weak (expected for current SimCLR) |
| 0.50–0.75 | Usable as a top-8 candidate list |
| > 0.75   | Ship it |

In [ ]:
import json, pathlib

def load_report(path):
    with open(path) as f:
        return json.load(f)

dino_report = load_report(f"Tools/dinov2-embeddings/reports/{MODEL}.json")
simclr_path = pathlib.Path("Tools/dinov2-embeddings/reports/simclr_shipped.json")

print(f"=== DINOv2 ({MODEL}) ===")
for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
    print(f"  {k}: {dino_report.get(k, 'N/A')}")

if simclr_path.exists():
    simclr_report = load_report(simclr_path)
    print(f"\n=== SimCLR (shipped) ===")
    for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
        print(f"  {k}: {simclr_report.get(k, 'N/A')}")
else:
    print("\nSimCLR report not found — skipping comparison.")

## 7. Inspect failures

The `sample_failures` field lists the 20 worst cases. If the top-5 for a failed query is full of figures with the same torso color but different prints, the encoder is confusing "color" for "identity" — try a stronger backbone or fine-tuning. If the top-5 looks random, there's a preprocessing mismatch.

In [ ]:
failures = dino_report.get("sample_failures", [])
print(f"{len(failures)} sample failures:\n")
for i, f in enumerate(failures[:10], 1):
    print(f"{i}. {f}")

## 8. Real-Photo Evaluation

Score the index against 27 real iPhone photos of LEGO minifigures (wood-grain background, shadows, etc.) instead of synthetic variants. This tests the domain gap between catalog renders and real-world captures.

In [ ]:
# The mapping.json was pre-generated from images/figurines/ filenames.
# The eval ground-truth set copies the photos into the evaluate_retrieval.py format.
!python Tools/dinov2-embeddings/ingest_real_photos.py eval \
    --mapping Tools/dinov2-embeddings/real_photos/mapping.json

!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/${MODEL} \
    --eval Tools/dinov2-embeddings/real_photos/eval \
    --report Tools/dinov2-embeddings/reports/${MODEL}_real_photos.json

In [ ]:
real_report = load_report(f"Tools/dinov2-embeddings/reports/{MODEL}_real_photos.json")
print(f"=== Real Photos vs {MODEL} (synthetic eval) ===")
print(f"{'Metric':<12} {'Synthetic':>10} {'Real Photo':>10}")
print("-" * 34)
for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
    synth = dino_report.get(k, "N/A")
    real = real_report.get(k, "N/A")
    synth_s = f"{synth:.3f}" if isinstance(synth, float) else str(synth)
    real_s = f"{real:.3f}" if isinstance(real, float) else str(real)
    print(f"{k:<12} {synth_s:>10} {real_s:>10}")

## 9. Augment Index with Real Photos

Add DINOv2 embeddings of the real photos to the index so the same figure has both a catalog render vector AND a real-world photo vector. This should improve recall when users scan real figures.

In [ ]:
!python Tools/dinov2-embeddings/ingest_real_photos.py augment \
    --mapping Tools/dinov2-embeddings/real_photos/mapping.json \
    --index Tools/dinov2-embeddings/index/${MODEL} \
    --out Tools/dinov2-embeddings/index/${MODEL}_augmented

In [ ]:
# Re-evaluate with the augmented index
!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/${MODEL}_augmented \
    --eval Tools/dinov2-embeddings/real_photos/eval \
    --report Tools/dinov2-embeddings/reports/${MODEL}_augmented_real.json

aug_report = load_report(f"Tools/dinov2-embeddings/reports/{MODEL}_augmented_real.json")
print(f"\n=== Impact of Augmentation (real-photo queries) ===")
print(f"{'Metric':<12} {'Before':>10} {'After':>10} {'Delta':>10}")
print("-" * 45)
for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
    before = real_report.get(k, 0)
    after = aug_report.get(k, 0)
    if isinstance(before, (int, float)) and isinstance(after, (int, float)):
        delta = after - before
        print(f"{k:<12} {before:>10.3f} {after:>10.3f} {delta:>+10.3f}")
    else:
        print(f"{k:<12} {before!s:>10} {after!s:>10}")

## 10. Cross-Source Eval: BrickLink vs Rebrickable

The index is built from **Rebrickable** renders. This section evaluates retrieval using **BrickLink** renders as queries — a completely different rendering pipeline (different camera angle, lighting, head color, aspect ratio). This tests whether the encoder generalises across render styles, which is a much harder and more realistic test than same-source synthetic augmentation.

**Prerequisites:**
- A Rebrickable API key (free at https://rebrickable.com/api/)
- Run `fetch_bricklink_images.py fetch` to download BrickLink renders
- Run `fetch_bricklink_images.py eval` to build the eval set

In [ ]:
# Install cloudscraper (needed for Rebrickable page scraping)
!pip install -q cloudscraper

# Step 1: Fetch BrickLink renders (200 figures, ~1-2 min)
!python Tools/dinov2-embeddings/fetch_bricklink_images.py fetch \
    --figures 200 --seed 42 \
    --out Tools/dinov2-embeddings/bricklink_images/

# Step 2: Build cross-source eval set
!python Tools/dinov2-embeddings/fetch_bricklink_images.py eval \
    --images Tools/dinov2-embeddings/bricklink_images/ \
    --out Tools/dinov2-embeddings/eval_bricklink/

In [ ]:
# Step 3: Run cross-source retrieval eval
!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/${MODEL} \
    --eval Tools/dinov2-embeddings/eval_bricklink/ \
    --report Tools/dinov2-embeddings/reports/${MODEL}_cross_source.json

cross_report = load_report(f"Tools/dinov2-embeddings/reports/{MODEL}_cross_source.json")
print(f"\n=== Cross-Source Eval: BrickLink queries → Rebrickable index ===")
print(f"{'Metric':<12} {'Synthetic':>10} {'CrossSrc':>10} {'Delta':>10}")
print("-" * 45)
for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
    synth = best_report.get(k, 0) if 'best_report' in dir() else 0
    cross = cross_report.get(k, 0)
    if isinstance(synth, (int, float)) and isinstance(cross, (int, float)):
        delta = cross - synth
        print(f"{k:<12} {synth:>10.3f} {cross:>10.3f} {delta:>+10.3f}")
    else:
        print(f"{k:<12} {synth!s:>10} {cross!s:>10}")

In [ ]:
# Step 4: Visual comparison — same figure, two render sources
from PIL import Image
from IPython.display import display
import json, random

bl_index = json.loads(open("Tools/dinov2-embeddings/bricklink_images/index.json").read())
sample_ids = random.Random(42).sample(list(bl_index.keys()), min(4, len(bl_index)))

for fid in sample_ids:
    rb_path = f"Bricky/Resources/MinifigImages/{fid}.jpg"
    bl_path = f"Tools/dinov2-embeddings/bricklink_images/{fid}.png"
    try:
        rb = Image.open(rb_path).convert("RGB")
        bl = Image.open(bl_path).convert("RGB")
        # Side-by-side
        h = max(rb.height, bl.height)
        canvas = Image.new("RGB", (rb.width + bl.width + 20, h + 30), (255, 255, 255))
        canvas.paste(rb, (0, 30 + (h - rb.height) // 2))
        canvas.paste(bl, (rb.width + 20, 30 + (h - bl.height) // 2))
        print(f"\n{fid} — Left: Rebrickable | Right: BrickLink")
        display(canvas)
    except FileNotFoundError:
        print(f"  {fid}: image not found, skipping")

<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/dinov2-embeddings/dinov2_retrieval_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>